# Learn OPC UA with the raw Python client

Connect to the background Mettler Toledo WXS205SDU simulator from a Python notebook.
Run the cells in order, then rerun individual operations to experiment.

| Setting | Value |
| --- | --- |
| Endpoint | `opc.tcp://127.0.0.1:4840/pylabrobot/mettler-toledo/` |
| Simulated serial number | `SIMULATED-001` |
| Initial weight | 12.345 g |
| Weight unit | grams |
| Connection | Localhost, anonymous, unencrypted |

Choose this repository's `env/bin/python` as your notebook kernel. If `asyncua` is
missing, install `PyLabRobot[opcua]` in that kernel's environment. The server must
already be running; this notebook only creates a client.

The connection cell checks that the endpoint identifies itself as the simulator
before allowing any measurement or control calls. This notebook is for the
simulator, not a hardware run. The OPC UA interface has not been verified on real
hardware. See [the server guide](opcua.md) for the interface details.

Use top-level `await` in notebook cells; no `asyncio.run()` is needed.

This notebook deliberately uses `asyncua.Client` directly. You will see namespace
indices, browse paths, NodeIds, method metadata, and status codes. The separate
[typed-client notebook](opcua-client-typed.ipynb) shows how `BalanceClient` hides
these details.

## What companion specifications add

OPC UA defines the common mechanisms: Objects, Variables, Methods, types,
references, subscriptions, and data quality. A **companion specification** defines
an agreed device model using those mechanisms, so vendors can expose the same
concepts consistently.

Two relevant published examples are:

- [OPC UA for Weighing Technology, OPC 40200](https://reference.opcfoundation.org/Scales/v200/docs/6.2):
  defines scale systems and scale-device types. Its scope includes laboratory scales.
- [LADS, OPC 30500-1](https://reference.opcfoundation.org/specs/OPC-30500-1/6):
  defines laboratory-device and functional-unit models, with device states and health.

Conformance means implementing the relevant specification's actual types,
mandatory members, state semantics, and profiles. Naming a variable `Weight` does
not establish conformance. This example retains its own PyLabRobot namespace and
is **not a Scales or LADS implementation**. It demonstrates their building blocks:
a core OPC UA `AnalogItemType`, units, measurement quality, and subscriptions.

The running simulator exposes:

```text
Objects
├── Balance                          PyLabRobot balance namespace
│   ├── Weight                       AnalogItemType; scalar Double
│   │   ├── 0:EngineeringUnits       EUInformation: gram (g)
│   │   └── 0:EURange                nominal net range: -capacity … +capacity
│   ├── DeviceState                  idle / busy / fault / stopped
│   ├── Fault                        latest operation's error text, or empty
│   ├── FaultCode                    latest operation's OPC UA StatusCode
│   └── …                            identity properties and methods
└── Simulation                       separate simulator-only namespace
    ├── SetLoad(load: Double)         set gross load in grams
    └── SetFault(active: Boolean)     inject or clear an overload
```

`DeviceState` describes what the adapter observes, not a firmware or LADS state
machine. Fault variables report the latest completed operation, not latched alarms;
this example does not implement OPC UA Alarms & Conditions.

## Configure the client

The namespace URI identifies the interface. Its numeric index is discovered after
connecting. A 70-second client timeout accommodates the driver's 60-second timeout;
queued commands can require longer.

In [1]:
import asyncio

from asyncua import Client, ua

ENDPOINT = "opc.tcp://127.0.0.1:4840/pylabrobot/mettler-toledo/"
NAMESPACE_URI = "https://pylabrobot.org/opcua/mettler-toledo/scales"

client = Client(ENDPOINT, timeout=70)

## Connect and locate the simulated balance

Run this once per session. If you need to reconnect, run the disconnect cell first,
then rerun the configuration and connection cells. A refused connection usually
means the background server is no longer running.

In [2]:
await client.connect()
try:
  namespace = await client.get_namespace_index(NAMESPACE_URI)
  balance = await client.nodes.objects.get_child(f"{namespace}:Balance")
  serial_node = await balance.get_child(f"{namespace}:SerialNumber")
  serial_number = await serial_node.read_value()
  if serial_number != "SIMULATED-001":
    raise RuntimeError(f"Expected the simulator; found serial number {serial_number!r}")
except BaseException:
  await client.disconnect()
  raise

print(f"Connected to {serial_number}, namespace index {namespace}")

Connected to SIMULATED-001, namespace index 2


## Inspect the namespace table

A namespace **URI** identifies a naming domain; a namespace **index** is its
position in this server's namespace table. Index 0 belongs to the OPC UA standard.
Our balance URI can have a different index on another server or after a restart,
which is why the connection cell resolves the URI instead of assuming `ns=2`.

A NodeId such as `ns=2;s=Balance.ReadStableWeight` identifies a node: `ns` is the
namespace index and `s` means a string identifier. A browse path component such
as `2:ReadStableWeight` is a qualified browse name used to navigate references;
it is not the same thing as a NodeId.

In [3]:
for index, uri in enumerate(await client.get_namespace_array()):
  print(f"ns={index}: {uri}")
print(f"Balance object NodeId: {balance.nodeid.to_string()}")

ns=0: http://opcfoundation.org/UA/
ns=1: urn:freeopcua:python:server
ns=2: https://pylabrobot.org/opcua/mettler-toledo/scales
ns=3: https://pylabrobot.org/opcua/mettler-toledo/simulation
Balance object NodeId: ns=2;s=Balance


## Browse the interface

Inspect each child node's browse name, node class, and stable NodeId. Methods perform
operations; properties describe the balance. `Weight` is a subscribable measurement variable. The sections below compare
explicit method calls with server-pushed data-change notifications.

In [4]:
for node in await balance.get_children():
  name = await node.read_browse_name()
  node_class = await node.read_node_class()
  print(f"{name.Name:24} {node_class.name:10} {node.nodeid.to_string()}")

DeviceType               Variable   ns=2;s=Balance.DeviceType
SerialNumber             Variable   ns=2;s=Balance.SerialNumber
FirmwareVersion          Variable   ns=2;s=Balance.FirmwareVersion
Unit                     Variable   ns=2;s=Balance.Unit
Capacity                 Variable   ns=2;s=Balance.Capacity
Weight                   Variable   ns=2;s=Balance.Weight
DeviceState              Variable   ns=2;s=Balance.DeviceState
Fault                    Variable   ns=2;s=Balance.Fault
FaultCode                Variable   ns=2;s=Balance.FaultCode
ReadStableWeight         Method     ns=2;s=Balance.ReadStableWeight
ReadWeightImmediately    Method     ns=2;s=Balance.ReadWeightImmediately
ReadTareWeight           Method     ns=2;s=Balance.ReadTareWeight
Zero                     Method     ns=2;s=Balance.Zero
Tare                     Method     ns=2;s=Balance.Tare
ClearTare                Method     ns=2;s=Balance.ClearTare


## Select a method and read its attributes

Change `method_name` to `Tare`, `ReadWeightImmediately`, or another method discovered
above, then rerun this cell and the argument-inspection cell.

A Method node has attributes such as `Description`, `Executable`, and
`UserExecutable`. The latter two advertise availability; the actual call can still
fail if the device or connection is not ready. A description is only as informative
as the server makes it; this server's method descriptions currently contain their
names.

In [5]:
method_name = "ReadStableWeight"
method = await balance.get_child(f"{namespace}:{method_name}")

print(f"NodeId: {method.nodeid.to_string()}")
print(f"Description: {(await method.read_description()).Text}")
for attribute in (ua.AttributeIds.Executable, ua.AttributeIds.UserExecutable):
  result = await method.read_attribute(attribute)
  print(f"{attribute.name}: {result.Value.Value}")

NodeId: ns=2;s=Balance.ReadStableWeight
Description: ReadStableWeight
Executable: True
UserExecutable: True


## Discover its parameters and return values

Method signatures live in the `InputArguments` and `OutputArguments` **properties**.
Their browse names are in namespace **0**, even though the Method node belongs to
our application namespace. If you know a property exists, access it directly with
`await method.get_child("0:OutputArguments")`, then read its value.

The code below browses the properties first so it also handles methods without
inputs or outputs. Each property contains an ordered list of `ua.Argument` values:

| Field | Meaning |
| --- | --- |
| `Name` | Parameter or return-value name, if provided by the server |
| `DataType` | NodeId of a type; browse that node to get a name such as `Double` |
| `ValueRank` | `-1`: scalar; `1`: one-dimensional array; positive `n`: `n` dimensions |
| `ArrayDimensions` | Maximum sizes for array dimensions, when specified |
| `Description` | Localized documentation text, if provided |

`ValueRank` also supports `0` (one or more dimensions), `-2` (any shape), and `-3`
(scalar or one-dimensional array). Missing names or descriptions mean the server
has not supplied them; clients cannot infer their meaning from the type alone.

All six methods in this example take **no input parameters**, so their
`InputArguments` properties are absent. The three read methods each return one
scalar `Double`; `Zero`, `Tare`, and `ClearTare` return no values. Output argument
names and descriptions have not been populated by this server.

References: [OPC UA Methods](https://reference.opcfoundation.org/specs/OPC-10000-3/5.7)
and [Argument fields](https://reference.opcfoundation.org/Core/Part3/v105/docs/8.6).

In [6]:
properties = {}
for prop in await method.get_properties():
  browse_name = await prop.read_browse_name()
  properties[(browse_name.NamespaceIndex, browse_name.Name)] = prop

for property_name in ("InputArguments", "OutputArguments"):
  prop = properties.get((0, property_name))
  arguments = [] if prop is None else await prop.read_value()
  print(f"{property_name}: {len(arguments)}")
  for index, argument in enumerate(arguments):
    type_node = client.get_node(argument.DataType)
    type_name = (await type_node.read_browse_name()).Name
    print(f"  [{index}] {argument.Name or '(unnamed)'}: {type_name}")
    print(f"      DataType NodeId: {argument.DataType.to_string()}")
    print(f"      ValueRank: {argument.ValueRank}")
    print(f"      ArrayDimensions: {argument.ArrayDimensions}")
    print(f"      Description: {argument.Description.Text or '(not supplied)'}")

InputArguments: 0
OutputArguments: 1
  [0] (unnamed): Double
      DataType NodeId: i=11
      ValueRank: -1
      ArrayDimensions: []
      Description: (not supplied)


## Read a raw OPC UA method result

A call identifies both the owning **ObjectId** and the **MethodId**. Input values
are ordered `ua.Variant` objects matching the advertised `InputArguments` list.
This method takes no inputs, so the list is empty.

For a method that *did* advertise a scalar Double input, you would supply
`ua.Variant(1.5, ua.VariantType.Double)` at the corresponding position. Do not add
that value to this call: our method has no input arguments.

This low-level request exposes the status and typed output values. The next
measurement cells use `balance.call_method(...)`, which performs these steps and
converts the output to Python values for you.

In [7]:
read_method = await balance.get_child(f"{namespace}:ReadStableWeight")
request = ua.CallMethodRequest(
  ObjectId=balance.nodeid,
  MethodId=read_method.nodeid,
  InputArguments=[],
)
result = (await client.uaclient.call([request]))[0]
print(f"Status: {result.StatusCode}")
result.StatusCode.check()
for output in result.OutputArguments:
  print(f"Wire type: {output.VariantType.name}; Python value: {output.Value}")

Status: StatusCode(value=0)
Wire type: Double; Python value: 12.345


## Read device properties

These properties are read-only. `Capacity` is in grams; the other values are strings.

In [8]:
identity = {}
for name in ("DeviceType", "SerialNumber", "FirmwareVersion", "Capacity", "Unit"):
  node = await balance.get_child(f"{namespace}:{name}")
  identity[name] = await node.read_value()
identity

{'DeviceType': 'WXS205SDU (SIMULATED)',
 'SerialNumber': 'SIMULATED-001',
 'FirmwareVersion': 'simulation',
 'Capacity': 220.009,
 'Unit': 'g'}

## Read a stable weight

A fresh simulator returns **12.345 g**. Later readings reflect any tare or zero
operations performed by you or another client.

In [9]:
weight = await balance.call_method(f"{namespace}:ReadStableWeight")
print(f"Stable weight: {weight:.6f} g")

Stable weight: 12.345000 g


## Read immediately

This method does not wait for stability. The simulator returns the same fixed load;
a real balance can return a fluctuating measurement.

In [10]:
weight = await balance.call_method(f"{namespace}:ReadWeightImmediately")
print(f"Immediate weight: {weight:.6f} g")

Immediate weight: 12.345000 g


## Collect a short series

Change `sample_count` and `interval` to experiment. The simulator has a fixed gross
load, so repeated readings stay constant unless a client changes tare or zero.
This loop makes explicit requests. Later sections subscribe to `Weight` instead.

In [11]:
sample_count = 5
interval = 0.2
samples = []

for index in range(sample_count):
  weight = await balance.call_method(f"{namespace}:ReadWeightImmediately")
  samples.append(weight)
  print(f"{index + 1}: {weight:.6f} g")
  if index + 1 < sample_count:
    await asyncio.sleep(interval)

1: 12.345000 g
2: 12.345000 g
3: 12.345000 g
4: 12.345000 g
5: 12.345000 g


## Tare the current load

Taring stores the current load as the tare weight. The net reading becomes zero.
The next cell inspects that state.

In [12]:
await balance.call_method(f"{namespace}:Tare")

## Inspect tare and net weight

On a fresh simulator, expect a tare weight of 12.345 g and a net weight of 0 g.

In [13]:
tare = await balance.call_method(f"{namespace}:ReadTareWeight")
net = await balance.call_method(f"{namespace}:ReadStableWeight")
print(f"Tare: {tare:.6f} g; net: {net:.6f} g")

Tare: 12.345000 g; net: 0.000000 g


## Clear tare

Clearing tare removes the stored tare weight. It does not undo a zero operation.

In [14]:
await balance.call_method(f"{namespace}:ClearTare")

## Read the restored weight

If the simulator has not been zeroed, the reading returns to 12.345 g.

In [15]:
weight = await balance.call_method(f"{namespace}:ReadStableWeight")
print(f"After clearing tare: {weight:.6f} g")

After clearing tare: 12.345000 g


## Browse the standard analog measurement metadata

The generic client follows `HasTypeDefinition` to learn that `Weight` is an
[AnalogItemType](https://reference.opcfoundation.org/Core/Part8/v105/docs/5.3.2).
`EngineeringUnits` and `EURange` have standard namespace-0 browse names. Units use
the OPC UA `EUInformation` structure and the UN/CEFACT gram code, not just a custom
text label. The nominal range is useful for display and percentage deadbands; it
is not a calibration or accuracy claim.

In [16]:
weight_node = await balance.get_child(f"{namespace}:Weight")
state_node = await balance.get_child(f"{namespace}:DeviceState")
fault_node = await balance.get_child(f"{namespace}:Fault")
fault_code_node = await balance.get_child(f"{namespace}:FaultCode")

type_id = await weight_node.read_type_definition()
type_name = (await client.get_node(type_id).read_browse_name()).Name
units = await (await weight_node.get_child("0:EngineeringUnits")).read_value()
limits = await (await weight_node.get_child("0:EURange")).read_value()
print(f"Type: {type_name} ({type_id.to_string()})")
print(f"Units: {units.DisplayName.Text}; {units.Description.Text}; UnitId={units.UnitId}")
print(f"Nominal net range: {limits.Low} to {limits.High} {units.DisplayName.Text}")

Type: AnalogItemType (i=2368)
Units: g; gram; UnitId=4674125
Nominal net range: -220.009 to 220.009 g


## Read the value, quality, and timestamp together

`read_value()` returns the Python value and raises on bad quality.
`read_data_value(raise_on_bad_status=False)` lets you inspect the full OPC UA
`DataValue`, including a fault without raising.

Here `SourceTimestamp` is when the adapter received the measurement, not a timestamp
from the scale firmware. `Good` means the read succeeded; an immediate reading can
still be physically unstable. Before the first reading the status is
`BadWaitingForInitialData`. A failed read publishes a bad status and a null value,
so an old sample is not silently presented as a new good measurement.

In [17]:
sample = await weight_node.read_data_value(raise_on_bad_status=False)
print(f"Value: {sample.Value.Value}")
print(f"Quality: {sample.StatusCode.name}")
print(f"Source timestamp: {sample.SourceTimestamp}")
print(f"Server timestamp: {sample.ServerTimestamp}")

Value: 12.345
Quality: Good
Source timestamp: 2026-10-01 10:22:40.953411+00:00
Server timestamp: 2026-10-01 10:22:40.953429+00:00


## Subscribe with a generic raw client

The server samples immediate net weight every 0.5 seconds in this simulator. A
subscription delivers value or quality changes without a client-side read loop.
The requested 100 ms publishing interval controls notification delivery, not the
hardware sampling rate. Repeated equal values normally do not trigger a data
change just because their timestamp changed.

The callback keeps full data values, so bad-quality notifications retain their
status and timestamps. Subscribe to each node individually so subscription errors
are raised immediately. A queue of 20 retains short state transitions.

In [ ]:
from typing import Any

from asyncua import Node

monitored = {
  "Weight": weight_node,
  "DeviceState": state_node,
  "Fault": fault_node,
  "FaultCode": fault_code_node,
}
labels = {node.nodeid.to_string(): name for name, node in monitored.items()}


class Monitor:
  """Receive data changes without calling device methods or polling from the client."""

  def __init__(self) -> None:
    """Keep the latest DataValue for each node and a notification history."""
    self.latest: dict[str, ua.DataValue] = {}
    self.events: list[tuple[str, object, str]] = []

  def datachange_notification(self, node: Node, value: object, data: Any) -> None:
    """Record server-pushed values, including quality changes with a null value."""
    name = labels[node.nodeid.to_string()]
    sample = data.monitored_item.Value
    self.latest[name] = sample
    self.events.append((name, value, sample.StatusCode.name))
    print(f"{name}: {value} [{sample.StatusCode.name}]")


monitor = Monitor()
subscription = await client.create_subscription(100, monitor)
handles = []
for node in monitored.values():
  handles.append(await subscription.subscribe_data_change(node, queuesize=20))
await asyncio.sleep(0.3)  # Allow initial notifications to arrive.

Weight: 12.345 [Good]
DeviceState: idle [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
Fault:  [Good]
FaultCode: StatusCode(value=0) [Good]


DeviceState: busy [Good]
DeviceState: idle [Good]
Weight: 25.0 [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
DeviceState: busy [Good]
DeviceState: fault [Good]
Weight: None [BadDeviceFailure]
Fault: Balance in overload range.: None [Good]
FaultCode: StatusCode(value=2156593152) [Good]
DeviceState: busy [Good]
DeviceState: fault [Good]
DeviceState: busy [Good]
DeviceState: fault [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
Weight: 25.0 [Good]
Fault:  [Good]
FaultCode: StatusCode(value=0) [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]
Weight: 12.345 [Good]
DeviceState: busy [Good]
DeviceState: idle [Good]


## Inspect a method that actually has an input parameter

The six balance methods have no inputs. The simulator's `SetLoad` method provides
a concrete parameter example. It belongs to a separate namespace and is never
exposed by the hardware adapter. Its `InputArguments` property describes the name,
type, shape, and meaning of its `load` parameter.

In [19]:
simulation_namespace = await client.get_namespace_index(
  "https://pylabrobot.org/opcua/mettler-toledo/simulation"
)
simulation = await client.nodes.objects.get_child(f"{simulation_namespace}:Simulation")
set_load = await simulation.get_child(f"{simulation_namespace}:SetLoad")
inputs = await (await set_load.get_child("0:InputArguments")).read_value()
for argument in inputs:
  data_type = (await client.get_node(argument.DataType).read_browse_name()).Name
  print(f"{argument.Name}: {data_type}, ValueRank={argument.ValueRank}")
  print(argument.Description.Text)

load: Double, ValueRank=-1
Simulated gross load in grams, from 0 to 220.009


## Change the load and watch the notification

Pass a scalar `Double` matching the advertised input type. The next server poll
updates `Weight`, and the subscription prints the change. No weight method is
called in this cell. The wait gives the 0.5-second poll and 100 ms subscription
time to deliver the update; adjust it if you configure a slower server.

In [20]:
await simulation.call_method(set_load, ua.Variant(25.0, ua.VariantType.Double))
await asyncio.sleep(1.5)
print("Latest subscribed weight:", monitor.latest["Weight"].Value.Value)

Latest subscribed weight: 25.0


## Inject a fault and inspect data quality

`SetFault(True)` makes simulated balance commands report overload. Observe the
`DeviceState`, `Fault`, and `FaultCode` notifications, plus bad quality on `Weight`.
The `FaultCode` variable contains a status code as its value; its own DataValue can
still be Good because the server successfully delivered the fault information.

Run the recovery cell after this one. While the fault is active, the server keeps
sampling, and a direct weight-method call would raise an OPC UA error.

In [21]:
await simulation.call_method(f"{simulation_namespace}:SetFault", True)
await asyncio.sleep(1.5)
bad_sample = monitor.latest["Weight"]
print(f"Weight: {bad_sample.Value.Value}; quality: {bad_sample.StatusCode.name}")
print("Observed state:", monitor.latest["DeviceState"].Value.Value)
print("Fault:", monitor.latest["Fault"].Value.Value)

Weight: None; quality: BadDeviceFailure
Observed state: fault
Fault: Balance in overload range.: None


## Recover and watch the weight become valid again

Clearing the simulated fault allows the next poll to succeed. The server clears
the fault text and code and publishes a Good weight value. This recovery does not
require a new subscription.

In [22]:
await simulation.call_method(f"{simulation_namespace}:SetFault", False)
await asyncio.sleep(1.5)
recovered = monitor.latest["Weight"]
print(f"Recovered: {recovered.Value.Value} g [{recovered.StatusCode.name}]")

Recovered: 25.0 g [Good]


## Restore the load and delete the subscription

Run this cleanup cell when finished, even if you interrupted an experiment. It
clears an injected fault, restores the 12.345 g gross load, and deletes this
subscription. The server and its polling task keep running. Existing zero/tare
references are not changed by setting the load.

In [23]:
try:
  await simulation.call_method(f"{simulation_namespace}:SetFault", False)
  await simulation.call_method(set_load, ua.Variant(12.345, ua.VariantType.Double))
  await asyncio.sleep(1.0)
finally:
  await subscription.delete()
print(f"Received {len(monitor.events)} notifications; subscription deleted.")

Received 36 notifications; subscription deleted.


## Inspect an OPC UA error

All six methods take no arguments. Deliberately send an extra argument to a read
method and catch `BadTooManyArguments`. The server rejects this request before
sending a balance command.

In [24]:
try:
  await balance.call_method(f"{namespace}:ReadStableWeight", 123)
except ua.UaStatusCodeError as error:
  print(f"OPC UA error: {error}")
  assert error.code == ua.StatusCodes.BadTooManyArguments

OPC UA error: Too many arguments were provided.(BadTooManyArguments)


## Optional: zero the balance

Set `run_zero` to `True` to try zeroing. It is off by default so **Run All** leaves
the simulator's initial load available for more experiments.

Zeroing sets the current load as the zero reference and clears tare. Subsequent
readings are 0 g. `ClearTare` will not restore 12.345 g after zeroing; restart the
simulator to reset its zero reference. Disconnecting this client does not reset it.

In [25]:
run_zero = False

if run_zero:
  await balance.call_method(f"{namespace}:Zero")
  print("Zeroed. Rerun a weight-reading cell to see the result.")
else:
  print("Zero skipped; set run_zero = True to try it.")

Zero skipped; set run_zero = True to try it.


## Disconnect

Run this when finished, including after an interrupted experiment. It closes only
this client session; the background server continues running.

In [26]:
await client.disconnect()
print("Client disconnected; the server is still running.")

Client disconnected; the server is still running.
